# Baseline Model Evaluation
**Binary Classification: Seizure vs Non-Seizure**

This notebook provides a detailed medical performance evaluation of the Baseline Logistic Regression model. 
It focuses on metrics critical for clinical diagnostics, such as **Sensitivity (Recall)** and the trade-off between **False Positives** and **False Negatives**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc
)

# Set style
sns.set(style="whitegrid")

## 1. Load Predictions
Loading the test set predictions and probabilities generated by `scripts/train_baseline.py`.

In [ ]:
y_true = np.load("../outputs/y_test_true.npy")
y_pred = np.load("../outputs/y_test_pred.npy")
y_prob = np.load("../outputs/y_test_prob.npy")

print(f"Loaded {len(y_true)} test samples.")

## 2. Confusion Matrix (Core Medical Metrics)
Analyzing the raw counts of True Positives, False Negatives, etc.

In [ ]:
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

print(f"True Positives (TP): {tp}")
print(f"False Negatives (FN): {fn}")
print(f"False Positives (FP): {fp}")
print(f"True Negatives (TN): {tn}")

# Visualization
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Non-Seizure", "Seizure"],
    yticklabels=["Non-Seizure", "Seizure"]
)
plt.title("Confusion Matrix – Baseline Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## 3. Precision, Recall, F1
**Recall (Sensitivity)** is the most critical metric. It tells us what percentage of actual seizures were detected.

In [ ]:
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Precision (Seizure): {precision:.3f}")
print(f"Recall / Sensitivity (Seizure): {recall:.3f}")
print(f"F1-score: {f1:.3f}")

## 4. Error Analysis: False Positive Rate vs False Negative Rate

In [ ]:
fp_rate = fp / (fp + tn)
fn_rate = fn / (fn + tp)

metrics_df = pd.DataFrame({
    "Metric": ["False Positive Rate", "False Negative Rate"],
    "Value": [fp_rate, fn_rate]
})

plt.figure(figsize=(6, 4))
sns.barplot(data=metrics_df, x="Metric", y="Value", palette="viridis")
plt.title("Error Type Comparison – Baseline Model")
plt.ylabel("Rate")
plt.ylim(0, max(fp_rate, fn_rate) + 0.05)
plt.grid(alpha=0.3, axis='y')
for index, row in metrics_df.iterrows():
    plt.text(index, row.Value + 0.005, f'{row.Value:.3f}', color='black', ha="center")
plt.tight_layout()
plt.show()

## 5. ROC Curve & AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}", color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], linestyle="--", color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve – Baseline Logistic Regression")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Threshold Tuning
Exploring how changing the decision threshold affects the Precision-Recall trade-off.

In [ ]:
thresholds_to_test = np.linspace(0.1, 0.9, 50)

results = []

for t in thresholds_to_test:
    y_thresh = (y_prob >= t).astype(int)
    results.append({
        "Threshold": t,
        "Precision": precision_score(y_true, y_thresh, zero_division=0),
        "Recall": recall_score(y_true, y_thresh)
    })

threshold_df = pd.DataFrame(results)

plt.figure(figsize=(8, 5))
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], label="Precision", color='green')
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], label="Recall", color='red')
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision–Recall Tradeoff")
plt.axvline(0.5, color='gray', linestyle='--', label='Default Threshold (0.5)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Summary & Conclusion

The baseline logistic regression model achieves high accuracy primarily due to strong amplitude-based separability between seizure and non-seizure segments. However, the recall of **89%** indicates that a non-negligible number of seizures are missed. 

This motivates the use of **deep learning models** capable of capturing temporal and morphological EEG patterns to further improve sensitivity without sacrificing precision.

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "Value": [
        (tp + tn) / (tp + tn + fp + fn),
        precision,
        recall,
        f1
    ]
})

summary